# Effect size and statistical summary


In [ ]:
import pandas as pd
import numpy as np
import json
import os
from IPython.display import display, HTML

# Paths
base_dir = '$PATH/RI_anatomical_prior/result_0619/JMRI'
data_root = '$PATH/RI_anatomical_prior/result_0619'

# Load existing statistical test results
ttest_df = pd.read_csv(os.path.join(base_dir, 'ttest_ind.csv'))
mannwhitney_df = pd.read_csv(os.path.join(base_dir, 'mannwhitneyu.csv'))
anova_df = pd.read_csv(os.path.join(base_dir, 'anova_results.csv'))
perf_summary = pd.read_csv(os.path.join(base_dir, 'option_performance_summary.csv'))
metric_stats = pd.read_csv(os.path.join(base_dir, 'metric_stats.csv'))

print("Performance Summary (option_performance_summary.csv):")
display(perf_summary)
print(f"\nMetric Stats shape: {metric_stats.shape}")
print(f"Models: {metric_stats['model'].unique()}")

## 1. Load per-seed metric data for Cohen's d calculation
We need the raw per-seed metrics (not just mean/std) to compute effect sizes accurately.

In [ ]:
"""
Load per-seed, per-model metrics from JSON files across all architectures.
Two experiment rounds: result_0619 (seeds 0-4) and result_0619_seed_1234 (seeds 0-4).
"""

root_paths = [
    '$PATH/RI_anatomical_prior/result_0619',
    '$PATH/RI_anatomical_prior/result_0619_seed_1234',
]

# Model directories: base models use None_None, _film models use freesurfer_both
model_dirs_base = ['agenet', 'basiccnn', 'cbr_tiny', 'chen_net', 'densenet', 'efficientnet', 'resnet18', 'resnet34', 'resnet50', 'sfcn', 'unet']
model_dirs_film = [m + '_film' for m in model_dirs_base]

options_base = ['t1_None_None', 't2_None_None', 'ratio_None_None', 't1t2_None_None', 't1t2ratio_None_None']
options_film = ['t1_freesurfer_both', 't2_freesurfer_both', 'ratio_freesurfer_both', 't1t2_freesurfer_both', 't1t2ratio_freesurfer_both']

seeds = [0, 1, 2, 3, 4]
metrics_list = ['mae', 'rmse', 'r2', 'pearsonr', 'spearmanr']

# Collect all per-seed metrics into a flat list
all_records = []

for root_path in root_paths:
    round_name = 'r1' if '0619_seed' not in root_path else 'r2'
    
    for model_dirs, options in [(model_dirs_base, options_base), (model_dirs_film, options_film)]:
        for model_dir in model_dirs:
            for option in options:
                for seed in seeds:
                    json_path = os.path.join(root_path, model_dir, f'{seed}_{option}_metrics.json')
                    if os.path.exists(json_path):
                        with open(json_path) as f:
                            data = json.load(f)
                        record = {
                            'model': model_dir,
                            'option': f'{option}_metrics',
                            'seed': seed,
                            'round': round_name,
                        }
                        for m in metrics_list:
                            record[m] = data.get(m, np.nan)
                        all_records.append(record)

raw_df = pd.DataFrame(all_records)
print(f"Total records loaded: {len(raw_df)}")
print(f"Models ({len(raw_df['model'].unique())}): {sorted(raw_df['model'].unique())}")
print(f"Options: {sorted(raw_df['option'].unique())}")
raw_df.head()

## 2. Modality label mapping & grouping by modality (across all models)

In [ ]:
def option_to_modality(option):
    """Convert option string to clean modality label."""
    base = option.split('_')[0].lower()
    mapping = {
        't1': 'T1w',
        't2': 'T2w',
        'ratio': 'RI',
        't1t2': 'T1w+T2w',
        't1t2ratio': 'T1w+T2w+RI',
    }
    return mapping.get(base, base)

raw_df['modality'] = raw_df['option'].apply(option_to_modality)

raw_df['prior'] = raw_df['option'].apply(lambda x: 'with_prior' if 'freesurfer' in x else 'without_prior')

print("Modality counts (without prior):")
print(raw_df[raw_df['prior'] == 'without_prior'].groupby('modality').size())
print("\nModality counts (with prior):")
print(raw_df[raw_df['prior'] == 'with_prior'].groupby('modality').size())

## 3. Mean ± SD Summary Table (all modalities)

In [ ]:
modality_order = ['T1w', 'T2w', 'T1w+T2w', 'RI', 'T1w+T2w+RI']

def make_summary_table(df_subset, label):
    """Create mean ± SD summary for each modality and metric."""
    rows = []
    for mod in modality_order:
        mod_data = df_subset[df_subset['modality'] == mod]
        if len(mod_data) == 0:
            continue
        row = {'Modality': mod, 'N': len(mod_data)}
        for metric in metrics_list:
            mean_val = mod_data[metric].mean()
            std_val = mod_data[metric].std()
            row[metric] = f"{mean_val:.4f} ± {std_val:.4f}"
        rows.append(row)
    result = pd.DataFrame(rows)
    print(f"\n{'='*80}")
    print(f"  {label}")
    print(f"{'='*80}")
    display(result)
    return result

summary_no_prior = make_summary_table(
    raw_df[raw_df['prior'] == 'without_prior'], 
    "Mean ± SD: Without Anatomical Prior"
)

summary_with_prior = make_summary_table(
    raw_df[raw_df['prior'] == 'with_prior'], 
    "Mean ± SD: With Anatomical Prior (FreeSurfer)"
)

## 4. Effect Size Calculations

### Cohen's d for pairwise comparisons (t-test, Mann-Whitney U)
- **Small**: |d| = 0.2, **Medium**: |d| = 0.5, **Large**: |d| = 0.8

### Eta-squared (η²) for ANOVA
- **Small**: η² = 0.01, **Medium**: η² = 0.06, **Large**: η² = 0.14

In [ ]:
from scipy import stats

def cohens_d(group1, group2):
    """Compute Cohen's d effect size between two groups."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    d = (group1.mean() - group2.mean()) / pooled_std
    return d

def interpret_cohens_d(d):
    """Interpret Cohen's d magnitude."""
    d_abs = abs(d)
    if d_abs < 0.2:
        return "negligible"
    elif d_abs < 0.5:
        return "small"
    elif d_abs < 0.8:
        return "medium"
    else:
        return "large"

def eta_squared_from_F(F_stat, df_between, df_within):
    """Compute eta-squared from F-statistic."""
    return (F_stat * df_between) / (F_stat * df_between + df_within)

def interpret_eta_squared(eta2):
    """Interpret eta-squared magnitude."""
    if eta2 < 0.01:
        return "negligible"
    elif eta2 < 0.06:
        return "small"
    elif eta2 < 0.14:
        return "medium"
    else:
        return "large"

print("Effect size functions defined.")

## 5. Pairwise Comparisons: Cohen's d + p-value (t-test & Mann-Whitney U)
Reference model: **RI** (ratio) — best model is compared against each other modality.

- Significant (p < 0.05): report Cohen's d only (no p-value)
- Non-significant (p ≥ 0.05): report Cohen's d + exact p-value

In [ ]:
def simplify_best_vs(label):
    """Convert best_vs column to clean modality label."""
    base = label.split('_')[0].lower()
    mapping = {
        't1': 'T1w',
        't2': 'T2w',
        'ratio': 'RI',
        't1t2': 'T1w+T2w',
        't1t2ratio': 'T1w+T2w+RI',
    }
    return mapping.get(base, base)

def has_freesurfer(label):
    return 'freesurfer' in label.lower()

def compute_pairwise_effect_sizes(test_df, test_name):
    """
    For each row in the test CSV (metric, best_vs, p_value),
    compute Cohen's d between the best model (RI without prior) and the comparison.
    """
    # Filter: exclude mse, exclude freesurfer comparisons
    df = test_df.copy()
    df = df[~df['metric'].isin(['mse'])]
    df = df[~df['best_vs'].str.contains('freesurfer', case=False, na=False)]
    df = df[df['p_value'].notna()]
    
    rows = []
    for _, row in df.iterrows():
        metric = row['metric']
        comparison_option = row['best_vs']  # e.g., t1t2_None_None_metrics
        p_val = row['p_value']
        significant = p_val < 0.05
        
        comparison_mod = simplify_best_vs(comparison_option)
        
        best_data = raw_df[(raw_df['option'] == 'ratio_None_None_metrics') & (raw_df['prior'] == 'without_prior')]
        comp_data = raw_df[(raw_df['option'] == comparison_option)]
        
        if len(best_data) == 0 or len(comp_data) == 0:
            continue
        
        d = cohens_d(best_data[metric], comp_data[metric])
        d_interp = interpret_cohens_d(d)
        
        result_row = {
            'Metric': metric.upper() if metric in ['mae', 'rmse'] else ('$R^2$' if metric == 'r2' else ('$r$' if metric == 'pearsonr' else '$\\rho$')),
            'Comparison': f"RI vs {comparison_mod}",
            'RI (mean±SD)': f"{best_data[metric].mean():.4f} ± {best_data[metric].std():.4f}",
            f'{comparison_mod} (mean±SD)': f"{comp_data[metric].mean():.4f} ± {comp_data[metric].std():.4f}",
            "Cohen's d": f"{d:.3f}",
            'Effect Size': d_interp,
            'Significant': 'Yes' if significant else 'No',
        }
        
        if not significant:
            result_row['p-value'] = f"{p_val:.4f}"
        else:
            result_row['p-value'] = '—'
        
        rows.append(result_row)
    
    result_df = pd.DataFrame(rows)
    return result_df

# T-test results
print("=" * 100)
print("  Independent t-test: Effect Size & Statistical Summary")
print("=" * 100)
ttest_effect = compute_pairwise_effect_sizes(ttest_df, 'ttest')
display(ttest_effect)

print("\n" + "=" * 100)
print("  Mann-Whitney U Test: Effect Size & Statistical Summary")
print("=" * 100)
mw_effect = compute_pairwise_effect_sizes(mannwhitney_df, 'mannwhitney')
display(mw_effect)

## 6. ANOVA: η² (Eta-squared) Effect Size

In [ ]:
"""
Compute eta-squared for ANOVA results.
The ANOVA was computed across all 10 options (5 modalities x 2 prior conditions).
"""

metric_label_map = {
    'mae': 'MAE',
    'rmse': 'RMSE',
    'r2': '$R^2$',
    'pearsonr': '$r$',
    'spearmanr': '$\\rho$'
}

# Number of groups in ANOVA (from included_options)
n_groups = 10  # 5 modalities x 2 (with/without prior)

anova_rows = []
for _, row in anova_df.iterrows():
    metric = row['metric']
    if metric == 'mse':
        continue
    
    F_stat = row['F_statistic']
    p_val = row['p_value']
    significant = p_val < 0.05
    
    # Get all data for this metric to compute df
    # Each group has varying N depending on available seeds/models
    options_in_anova = [o.strip() for o in row['included_options'].split(',')]
    
    group_sizes = []
    for opt in options_in_anova:
        n = len(raw_df[raw_df['option'] == opt])
        group_sizes.append(n)
    
    N_total = sum(group_sizes)
    k = len(options_in_anova)
    df_between = k - 1
    df_within = N_total - k
    
    eta2 = eta_squared_from_F(F_stat, df_between, df_within)
    eta2_interp = interpret_eta_squared(eta2)
    
    result_row = {
        'Metric': metric_label_map.get(metric, metric),
        'F-statistic': f"{F_stat:.4f}",
        'η²': f"{eta2:.4f}",
        'Effect Size': eta2_interp,
        'Significant': 'Yes' if significant else 'No',
        'k (groups)': k,
        'N (total)': N_total,
    }
    
    if not significant:
        result_row['p-value'] = f"{p_val:.4f}"
    else:
        result_row['p-value'] = '—'
    
    anova_rows.append(result_row)

anova_effect = pd.DataFrame(anova_rows)
print("=" * 100)
print("  One-Way ANOVA: η² Effect Size & Statistical Summary")
print("=" * 100)
display(anova_effect)

## 7. Comprehensive Pairwise Table: All modality pairs with Cohen's d (without prior)

In [ ]:
"""
Compute Cohen's d for ALL pairwise modality comparisons (without prior condition).
Also run independent t-test and Mann-Whitney U to get p-values.
"""

modality_to_option = {
    'T1w': 't1_None_None_metrics',
    'T2w': 't2_None_None_metrics',
    'T1w+T2w': 't1t2_None_None_metrics',
    'RI': 'ratio_None_None_metrics',
    'T1w+T2w+RI': 't1t2ratio_None_None_metrics',
}

from itertools import combinations

all_pairwise_rows = []

for metric in metrics_list:
    metric_label = metric_label_map.get(metric, metric)
    
    for mod_a, mod_b in combinations(modality_order, 2):
        opt_a = modality_to_option[mod_a]
        opt_b = modality_to_option[mod_b]
        
        data_a = raw_df[raw_df['option'] == opt_a][metric].dropna()
        data_b = raw_df[raw_df['option'] == opt_b][metric].dropna()
        
        if len(data_a) < 2 or len(data_b) < 2:
            continue
        
        # t-test
        t_stat, t_pval = stats.ttest_ind(data_a, data_b)
        # Mann-Whitney U
        u_stat, u_pval = stats.mannwhitneyu(data_a, data_b, alternative='two-sided')
        # Cohen's d
        d = cohens_d(data_a, data_b)
        d_interp = interpret_cohens_d(d)
        
        t_sig = t_pval < 0.05
        u_sig = u_pval < 0.05
        
        row = {
            'Metric': metric_label,
            'Comparison': f"{mod_a} vs {mod_b}",
            f'{mod_a} (mean±SD)': f"{data_a.mean():.4f} ± {data_a.std():.4f}",
            f'{mod_b} (mean±SD)': f"{data_b.mean():.4f} ± {data_b.std():.4f}",
            "Cohen's d": f"{d:.3f}",
            'Effect Size': d_interp,
            't-test sig.': 'Yes' if t_sig else 'No',
            't-test p': '—' if t_sig else f"{t_pval:.4f}",
            'MW-U sig.': 'Yes' if u_sig else 'No',
            'MW-U p': '—' if u_sig else f"{u_pval:.4f}",
        }
        all_pairwise_rows.append(row)

all_pairwise_df = pd.DataFrame(all_pairwise_rows)

# Display per metric
for metric in metrics_list:
    metric_label = metric_label_map.get(metric, metric)
    subset = all_pairwise_df[all_pairwise_df['Metric'] == metric_label]
    print(f"\n{'='*100}")
    print(f"  Pairwise Comparisons — {metric_label} (without prior)")
    print(f"{'='*100}")
    display(subset[['Comparison', "Cohen's d", 'Effect Size', 't-test sig.', 't-test p', 'MW-U sig.', 'MW-U p']].reset_index(drop=True))

## 8. Effect Size Visualization: Cohen's d Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, len(metrics_list), figsize=(5 * len(metrics_list), 4.5))

for idx, metric in enumerate(metrics_list):
    metric_label = metric_label_map.get(metric, metric)
    ax = axes[idx]
    
    # Build Cohen's d matrix
    d_matrix = pd.DataFrame(np.nan, index=modality_order, columns=modality_order)
    
    for mod_a in modality_order:
        for mod_b in modality_order:
            if mod_a == mod_b:
                d_matrix.loc[mod_a, mod_b] = 0.0
                continue
            opt_a = modality_to_option[mod_a]
            opt_b = modality_to_option[mod_b]
            data_a = raw_df[raw_df['option'] == opt_a][metric].dropna()
            data_b = raw_df[raw_df['option'] == opt_b][metric].dropna()
            if len(data_a) >= 2 and len(data_b) >= 2:
                d_matrix.loc[mod_a, mod_b] = cohens_d(data_a, data_b)
    
    d_matrix = d_matrix.astype(float)
    
    # Annotate with significance markers
    annot_matrix = d_matrix.copy().astype(str)
    for mod_a in modality_order:
        for mod_b in modality_order:
            if mod_a == mod_b:
                annot_matrix.loc[mod_a, mod_b] = "—"
                continue
            opt_a = modality_to_option[mod_a]
            opt_b = modality_to_option[mod_b]
            data_a = raw_df[raw_df['option'] == opt_a][metric].dropna()
            data_b = raw_df[raw_df['option'] == opt_b][metric].dropna()
            if len(data_a) >= 2 and len(data_b) >= 2:
                _, p = stats.ttest_ind(data_a, data_b)
                d_val = d_matrix.loc[mod_a, mod_b]
                sig_marker = "*" if p < 0.05 else ""
                annot_matrix.loc[mod_a, mod_b] = f"{d_val:.2f}{sig_marker}"
    
    sns.heatmap(
        d_matrix, ax=ax, annot=annot_matrix, fmt='',
        cmap='RdBu_r', center=0, vmin=-1.5, vmax=1.5,
        square=True, linewidths=0.5,
        cbar_kws={'shrink': 0.7, 'label': "Cohen's d"},
        annot_kws={'fontsize': 10}
    )
    ax.set_title(metric_label, fontsize=18)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.suptitle("Cohen's d Effect Size Matrix (without prior)\n* = significant (p < 0.05)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 9. Summary table (RI as reference)

In [ ]:
"""
Final paper-ready summary table.
For each metric, show:
- Mean ± SD for each modality
- Cohen's d (vs RI reference)
- Significance: if significant -> just mark, if not -> show exact p-value
"""

reference_mod = 'RI'
ref_option = modality_to_option[reference_mod]
comparison_mods = ['T1w', 'T2w', 'T1w+T2w', 'T1w+T2w+RI']

paper_rows = []

for metric in metrics_list:
    metric_label = metric_label_map.get(metric, metric)
    
    # Reference data
    ref_data = raw_df[raw_df['option'] == ref_option][metric].dropna()
    ref_mean = ref_data.mean()
    ref_std = ref_data.std()
    
    for comp_mod in comparison_mods:
        comp_option = modality_to_option[comp_mod]
        comp_data = raw_df[raw_df['option'] == comp_option][metric].dropna()
        comp_mean = comp_data.mean()
        comp_std = comp_data.std()
        
        # Cohen's d
        d = cohens_d(ref_data, comp_data)
        d_interp = interpret_cohens_d(d)
        
        # t-test
        _, t_p = stats.ttest_ind(ref_data, comp_data)
        # Mann-Whitney U
        _, u_p = stats.mannwhitneyu(ref_data, comp_data, alternative='two-sided')
        
        t_sig = t_p < 0.05
        u_sig = u_p < 0.05
        
        row = {
            'Metric': metric_label,
            'Comparison': f"RI vs {comp_mod}",
            'RI (mean ± SD)': f"{ref_mean:.4f} ± {ref_std:.4f}",
            f'{comp_mod} (mean ± SD)': f"{comp_mean:.4f} ± {comp_std:.4f}",
            "Cohen's d": f"{d:.3f} ({d_interp})",
        }
        
        # t-test: significant -> no p-value, non-significant -> exact p-value
        if t_sig:
            row['t-test'] = 'Significant'
        else:
            row['t-test'] = f'p = {t_p:.4f}'
        
        # Mann-Whitney: significant -> no p-value, non-significant -> exact p-value
        if u_sig:
            row['Mann-Whitney U'] = 'Significant'
        else:
            row['Mann-Whitney U'] = f'p = {u_p:.4f}'
        
        paper_rows.append(row)

paper_df = pd.DataFrame(paper_rows)

print("=" * 120)
print("  PAPER-READY TABLE: Effect Size & Statistical Significance (RI as reference)")
print("  - Significant results: marked without p-value")
print("  - Non-significant results: exact p-value shown")
print("=" * 120)

# Display grouped by metric
for metric in metrics_list:
    metric_label = metric_label_map.get(metric, metric)
    subset = paper_df[paper_df['Metric'] == metric_label].reset_index(drop=True)
    print(f"\n--- {metric_label} ---")
    display(subset)

## 10. Export to CSV

In [ ]:
output_dir = base_dir

# Save paper-ready table
paper_df.to_csv(os.path.join(output_dir, 'effect_size_pairwise_vs_RI.csv'), index=False)
print(f"Saved: {os.path.join(output_dir, 'effect_size_pairwise_vs_RI.csv')}")

# Save all pairwise comparisons
all_pairwise_df.to_csv(os.path.join(output_dir, 'effect_size_all_pairwise.csv'), index=False)
print(f"Saved: {os.path.join(output_dir, 'effect_size_all_pairwise.csv')}")

# Save ANOVA effect sizes
anova_effect.to_csv(os.path.join(output_dir, 'effect_size_anova.csv'), index=False)
print(f"Saved: {os.path.join(output_dir, 'effect_size_anova.csv')}")

print("\nDone!")

---
## 11. Forest Plot (Cohen's d ± 95% CI)
The most recommended visualization for effect sizes. Shows Cohen's d with confidence intervals relative to RI (reference).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

metric_label_map_forest = {
    'mae': 'MAE', 'rmse': 'RMSE', 'r2': '$R^2$',
    'pearsonr': '$r$', 'spearmanr': r'$\rho$'
}

comparison_mods = ['T1w', 'T2w', 'T1w+T2w', 'T1w+T2w+RI']
comp_colors = {'T1w': '#1f77b4', 'T2w': '#ff7f0e', 'T1w+T2w': '#2ca02c', 'T1w+T2w+RI': '#9467bd'}

ref_option = modality_to_option['RI']

def cohens_d_ci(g1, g2, alpha=0.05):
    """Cohen's d with 95% CI (Hedges & Olkin approximation)."""
    n1, n2 = len(g1), len(g2)
    d = cohens_d(g1, g2)
    se = np.sqrt((n1 + n2) / (n1 * n2) + d**2 / (2 * (n1 + n2)))
    z = stats.norm.ppf(1 - alpha / 2)
    return d, d - z * se, d + z * se

fig, axes = plt.subplots(1, len(metrics_list), figsize=(4.5 * len(metrics_list), 3.5), sharey=False)

for idx, metric in enumerate(metrics_list):
    ax = axes[idx]
    metric_label = metric_label_map_forest[metric]
    
    ref_data = raw_df[raw_df['option'] == ref_option][metric].dropna()
    
    y_positions = []
    y_labels = []
    
    for i, comp_mod in enumerate(comparison_mods):
        comp_option = modality_to_option[comp_mod]
        comp_data = raw_df[raw_df['option'] == comp_option][metric].dropna()
        
        d, ci_lo, ci_hi = cohens_d_ci(ref_data, comp_data)
        _, t_p = stats.ttest_ind(ref_data, comp_data)
        sig = t_p < 0.05
        
        y = len(comparison_mods) - 1 - i
        y_positions.append(y)
        y_labels.append(comp_mod)
        
        color = comp_colors[comp_mod]
        
        # CI line
        ax.plot([ci_lo, ci_hi], [y, y], color=color, linewidth=2.5, solid_capstyle='round')
        # CI caps
        ax.plot([ci_lo, ci_lo], [y - 0.1, y + 0.1], color=color, linewidth=2)
        ax.plot([ci_hi, ci_hi], [y - 0.1, y + 0.1], color=color, linewidth=2)
        # Point estimate
        marker = 'D' if sig else 'o'
        ax.plot(d, y, marker=marker, color=color, markersize=10, markeredgecolor='black', markeredgewidth=0.8, zorder=5)
        
        # Annotation
        if sig:
            ax.annotate(f'd = {d:.2f}', xy=(d, y), xytext=(8, 8),
                        textcoords='offset points', fontsize=12, fontweight='bold', color=color)
        else:
            ax.annotate(f'd = {d:.2f}\np = {t_p:.3f}', xy=(d, y), xytext=(8, 5),
                        textcoords='offset points', fontsize=11, color='gray')
    
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
    
    # Effect size regions
    ax.axvspan(-0.3, 0.2, alpha=0.05, color='gray')
    
    ax.set_yticks(y_positions)
    if idx == 0:
        ax.set_yticklabels(y_labels, fontsize=15)
    else:
        ax.set_yticklabels([])
    ax.set_xlabel("Cohen's d", fontsize=16)
    ax.set_title(metric_label, fontsize=22)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-0.5, 3.5)

# Title + Legend on same row at top
fig.text(0.5, 1.06, '(c) Forest Plot', fontsize=25, va='center', ha='center',
         transform=fig.transFigure)

diamond = plt.Line2D([], [], marker='D', color='gray', linestyle='None', markersize=8, markeredgecolor='black', label='Significant (p < 0.05)')
circle = plt.Line2D([], [], marker='o', color='gray', linestyle='None', markersize=8, markeredgecolor='black', label='Non-significant')
gray_patch = mpatches.Patch(facecolor='gray', alpha=0.15, label='Negligible effect (|d| < 0.2)')

fig.legend(handles=[diamond, circle, gray_patch], loc='upper right', ncol=3, fontsize=13,
           bbox_to_anchor=(0.99, 1.12), frameon=True, edgecolor='gray')

plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'fig_forest_plot_cohens_d.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(base_dir, 'fig_forest_plot_cohens_d.svg'), bbox_inches='tight')
plt.show()
print("Saved: fig_forest_plot_cohens_d.png / .svg") 

---
## 12. Bar Plot with Individual Data Points + Significance Brackets
Shows mean ± SD bars with jittered individual observations. Significant comparisons get asterisk brackets, non-significant get exact p-value.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

palette_bar = {'T1w': '#1f77b4', 'T2w': '#ff7f0e', 'T1w+T2w': '#2ca02c', 'RI': '#d62728', 'T1w+T2w+RI': '#9467bd'}

plot_data = raw_df[raw_df['prior'] == 'without_prior'].copy()

fig, axes = plt.subplots(1, len(metrics_list), figsize=(4.5 * len(metrics_list), 6))

for idx, metric in enumerate(metrics_list):
    ax = axes[idx]
    metric_label = metric_label_map_forest[metric]
    
    # Box + strip plot
    sns.boxplot(
        data=plot_data, x='modality', y=metric, order=modality_order,
        palette=palette_bar, ax=ax, width=0.6, fliersize=0,
        boxprops=dict(alpha=0.4), medianprops=dict(color='black', linewidth=2)
    )
    sns.stripplot(
        data=plot_data, x='modality', y=metric, order=modality_order,
        palette=palette_bar, ax=ax, size=3, alpha=0.5, jitter=0.2
    )
    
    ax.set_title(metric_label, fontsize=20, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(metric_label if idx == 0 else '', fontsize=14)
    ax.tick_params(axis='x', rotation=30, labelsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    ri_idx = modality_order.index('RI')
    ri_data = plot_data[plot_data['modality'] == 'RI'][metric].dropna()
    
    y_max = plot_data[metric].max()
    y_range = plot_data[metric].max() - plot_data[metric].min()
    bracket_y = y_max + y_range * 0.05
    step = y_range * 0.08
    
    bracket_count = 0
    for comp_mod in ['T1w', 'T2w', 'T1w+T2w', 'T1w+T2w+RI']:
        comp_idx = modality_order.index(comp_mod)
        comp_data = plot_data[plot_data['modality'] == comp_mod][metric].dropna()
        
        _, t_p = stats.ttest_ind(ri_data, comp_data)
        d = cohens_d(ri_data, comp_data)
        sig = t_p < 0.05
        
        by = bracket_y + bracket_count * step
        
        x1, x2 = min(ri_idx, comp_idx), max(ri_idx, comp_idx)
        
        # Draw bracket
        ax.plot([x1, x1, x2, x2], [by - step*0.1, by, by, by - step*0.1],
                color='black', linewidth=1, alpha=0.7)
        
        if sig:
            # Asterisks based on p-value
            if t_p < 0.001:
                star = '***'
            elif t_p < 0.01:
                star = '**'
            else:
                star = '*'
            ax.text((x1 + x2) / 2, by + step * 0.05, star,
                    ha='center', va='bottom', fontsize=12, fontweight='bold')
        else:
            ax.text((x1 + x2) / 2, by + step * 0.05, f'p={t_p:.3f}',
                    ha='center', va='bottom', fontsize=8, color='gray')
        
        bracket_count += 1
    
    # Extend y-axis to fit brackets
    ax.set_ylim(ax.get_ylim()[0], bracket_y + bracket_count * step + step * 0.5)

plt.suptitle('Performance by Modality (without prior) — RI as Reference', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'fig_barplot_with_significance.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(base_dir, 'fig_barplot_with_significance.svg'), bbox_inches='tight')
plt.show()
print("Saved: fig_barplot_with_significance.png / .svg")

---
## 13. Combined Summary Table (LaTeX-ready)
Clean publication table with Mean ± SD, Cohen's d, and significance status for each comparison.

In [ ]:
"""
Generate a clean summary table suitable for paper (LaTeX export).
Rows: Modality, Columns: each metric with mean±SD, then Cohen's d vs RI, significance.
"""

ref_option_tab = modality_to_option['RI']
ref_data_all = {}
for m in metrics_list:
    ref_data_all[m] = raw_df[raw_df['option'] == ref_option_tab][m].dropna()

table_rows = []
for mod in modality_order:
    opt = modality_to_option[mod]
    mod_data = raw_df[raw_df['option'] == opt]
    
    row = {'Modality': mod}
    
    for m in metrics_list:
        m_label = metric_label_map_forest[m]
        vals = mod_data[m].dropna()
        row[f'{m_label}'] = f"{vals.mean():.3f} ± {vals.std():.3f}"
        
        if mod != 'RI':
            d = cohens_d(ref_data_all[m], vals)
            _, t_p = stats.ttest_ind(ref_data_all[m], vals)
            sig = t_p < 0.05
            
            d_str = f"{d:.2f} ({interpret_cohens_d(d)[0].upper()}{interpret_cohens_d(d)[1:]})"
            
            if sig:
                if t_p < 0.001:
                    row[f'{m_label} (d / sig)'] = f"{d_str} ***"
                elif t_p < 0.01:
                    row[f'{m_label} (d / sig)'] = f"{d_str} **"
                else:
                    row[f'{m_label} (d / sig)'] = f"{d_str} *"
            else:
                row[f'{m_label} (d / sig)'] = f"{d_str} (p={t_p:.3f})"
        else:
            row[f'{m_label} (d / sig)'] = "Reference"
    
    table_rows.append(row)

summary_table = pd.DataFrame(table_rows)

# Display: Mean ± SD
print("=" * 100)
print("  Table 1: Performance Summary (Mean ± SD)")
print("=" * 100)
mean_cols = ['Modality'] + [metric_label_map_forest[m] for m in metrics_list]
display(summary_table[mean_cols])

# Display: Effect size & significance
print("\n" + "=" * 100)
print("  Table 2: Effect Size (Cohen's d vs RI) & Significance")
print("  * p<0.05, ** p<0.01, *** p<0.001 | Non-significant: exact p-value shown")
print("=" * 100)
effect_cols = ['Modality'] + [f'{metric_label_map_forest[m]} (d / sig)' for m in metrics_list]
display(summary_table[effect_cols])

# LaTeX export
latex_str = summary_table[mean_cols].to_latex(index=False, escape=False, column_format='l' + 'c' * len(metrics_list))
print("\n--- LaTeX (Mean ± SD) ---")
print(latex_str)

latex_str2 = summary_table[effect_cols].to_latex(index=False, escape=False, column_format='l' + 'c' * len(metrics_list))
print("\n--- LaTeX (Effect Size) ---")
print(latex_str2)

# Save
summary_table.to_csv(os.path.join(base_dir, 'paper_summary_table.csv'), index=False)
print(f"\nSaved: {os.path.join(base_dir, 'paper_summary_table.csv')}")

---
## 14. Combined figure

**Top row**: Forest Plot — Cohen's d ± 95% CI (effect size + significance at a glance)  
**Bottom row**: Box + Strip Plot — raw data distribution with significance brackets

This figure summarizes:
1. Mean ± SD (visible in boxplot)
2. Effect size (forest plot)
3. Exact p-value for non-significant (annotated)
4. No p-value for significant (only asterisks)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
from scipy import stats

# ── Settings ──
metrics_list_fig = ['mae', 'rmse', 'r2', 'pearsonr', 'spearmanr']
metric_labels = {'mae': 'MAE', 'rmse': 'RMSE', 'r2': '$R^2$', 'pearsonr': '$r$', 'spearmanr': r'$\rho$'}
comparison_mods = ['T1w', 'T2w', 'T1w+T2w', 'T1w+T2w+RI']
comp_colors = {'T1w': '#1f77b4', 'T2w': '#ff7f0e', 'T1w+T2w': '#2ca02c', 'T1w+T2w+RI': '#9467bd'}
palette_all = {'T1w': '#1f77b4', 'T2w': '#ff7f0e', 'T1w+T2w': '#2ca02c', 'RI': '#d62728', 'T1w+T2w+RI': '#9467bd'}

ref_opt = modality_to_option['RI']
plot_data = raw_df[raw_df['prior'] == 'without_prior'].copy()

n_metrics = len(metrics_list_fig)
fig = plt.figure(figsize=(4.2 * n_metrics, 10))
gs = gridspec.GridSpec(2, n_metrics, height_ratios=[1, 1.4], hspace=0.35)

# ═══════════════════════════════════════════════
# ═══════════════════════════════════════════════
for idx, metric in enumerate(metrics_list_fig):
    ax = fig.add_subplot(gs[0, idx])
    m_label = metric_labels[metric]
    
    ref_data = raw_df[raw_df['option'] == ref_opt][metric].dropna()
    
    for i, comp_mod in enumerate(comparison_mods):
        comp_opt = modality_to_option[comp_mod]
        comp_data = raw_df[raw_df['option'] == comp_opt][metric].dropna()
        
        d, ci_lo, ci_hi = cohens_d_ci(ref_data, comp_data)
        _, t_p = stats.ttest_ind(ref_data, comp_data)
        sig = t_p < 0.05
        
        y = len(comparison_mods) - 1 - i
        color = comp_colors[comp_mod]
        
        # CI line + caps
        ax.plot([ci_lo, ci_hi], [y, y], color=color, linewidth=2.5, solid_capstyle='round')
        ax.plot([ci_lo, ci_lo], [y - 0.08, y + 0.08], color=color, linewidth=1.8)
        ax.plot([ci_hi, ci_hi], [y - 0.08, y + 0.08], color=color, linewidth=1.8)
        
        # Point estimate
        marker = 'D' if sig else 'o'
        ax.plot(d, y, marker=marker, color=color, markersize=9,
                markeredgecolor='black', markeredgewidth=0.7, zorder=5)
        
        # Annotation
        if sig:
            ax.annotate(f'{d:.2f}', xy=(d, y), xytext=(7, 7),
                        textcoords='offset points', fontsize=9, fontweight='bold', color=color)
        else:
            ax.annotate(f'{d:.2f}\np={t_p:.3f}', xy=(d, y), xytext=(7, 4),
                        textcoords='offset points', fontsize=8, color='gray')
    
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.axvspan(-0.2, 0.2, alpha=0.06, color='gray')
    
    ax.set_yticks(range(len(comparison_mods)))
    if idx == 0:
        ax.set_yticklabels(list(reversed(comparison_mods)), fontsize=11)
    else:
        ax.set_yticklabels([])
    ax.set_xlabel("Cohen's d", fontsize=11)
    ax.set_title(m_label, fontsize=18, fontweight='bold', pad=10)
    ax.grid(axis='x', alpha=0.25)
    ax.set_xlim(-1.6, 1.6)

# ═══════════════════════════════════════════════
# BOTTOM ROW: Box + Strip Plot with brackets
# ═══════════════════════════════════════════════
for idx, metric in enumerate(metrics_list_fig):
    ax = fig.add_subplot(gs[1, idx])
    m_label = metric_labels[metric]
    
    sns.boxplot(
        data=plot_data, x='modality', y=metric, order=modality_order,
        palette=palette_all, ax=ax, width=0.55, fliersize=0,
        boxprops=dict(alpha=0.4), medianprops=dict(color='black', linewidth=1.8)
    )
    sns.stripplot(
        data=plot_data, x='modality', y=metric, order=modality_order,
        palette=palette_all, ax=ax, size=2.5, alpha=0.45, jitter=0.2
    )
    
    ax.set_title('')  # title already in top row
    ax.set_xlabel('')
    ax.set_ylabel(m_label if idx == 0 else '', fontsize=12)
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    ax.grid(axis='y', alpha=0.25)
    
    ri_idx = modality_order.index('RI')
    ri_data = plot_data[plot_data['modality'] == 'RI'][metric].dropna()
    
    y_max = plot_data[metric].max()
    y_range = plot_data[metric].max() - plot_data[metric].min()
    bracket_y = y_max + y_range * 0.05
    step = y_range * 0.07
    
    for bc, comp_mod in enumerate(['T1w', 'T2w', 'T1w+T2w', 'T1w+T2w+RI']):
        ci = modality_order.index(comp_mod)
        cd = plot_data[plot_data['modality'] == comp_mod][metric].dropna()
        _, tp = stats.ttest_ind(ri_data, cd)
        sig = tp < 0.05
        by = bracket_y + bc * step
        x1, x2 = min(ri_idx, ci), max(ri_idx, ci)
        
        ax.plot([x1, x1, x2, x2], [by - step*0.08, by, by, by - step*0.08],
                color='black', linewidth=0.8, alpha=0.6)
        
        if sig:
            star = '***' if tp < 0.001 else ('**' if tp < 0.01 else '*')
            ax.text((x1+x2)/2, by + step*0.03, star,
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
        else:
            ax.text((x1+x2)/2, by + step*0.03, f'p={tp:.3f}',
                    ha='center', va='bottom', fontsize=7, color='gray')
    
    ax.set_ylim(ax.get_ylim()[0], bracket_y + 4*step + step*0.4)

# ── Row labels ──
fig.text(0.01, 0.78, '(a)', fontsize=18, fontweight='bold', va='center')
fig.text(0.01, 0.32, '(b)', fontsize=18, fontweight='bold', va='center')

# ── Legend ──
diamond = plt.Line2D([], [], marker='D', color='gray', linestyle='None', markersize=7, markeredgecolor='black', label='Significant (p < 0.05)')
circle = plt.Line2D([], [], marker='o', color='gray', linestyle='None', markersize=7, markeredgecolor='black', label='Non-significant')
gray_patch = mpatches.Patch(facecolor='gray', alpha=0.15, label='|d| < 0.2 (negligible)')
fig.legend(handles=[diamond, circle, gray_patch], loc='upper center', ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, 1.01), frameon=True, edgecolor='gray')

plt.savefig(os.path.join(base_dir, 'fig_combined_effect_size.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(base_dir, 'fig_combined_effect_size.svg'), bbox_inches='tight')
plt.show()
print("Saved: fig_combined_effect_size.png / .svg")